<a href="https://colab.research.google.com/github/Heisenberg-hari/CIFPD/blob/main/another_copy_of_edaproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

In [ ]:
try:
    import geopandas as gpd
    GEO_ENABLED = True
except Exception:
    GEO_ENABLED = False

# ---------------------------
# 0. Config / file paths
# ---------------------------
DATA_DIR = '/content'
DEMAND_CSV = os.path.join(DATA_DIR, 'modified_agriculture.csv')
RAINFALL_CSV = os.path.join(DATA_DIR, 'rainfall_data 1.csv')
TEMP_CSV = os.path.join(DATA_DIR, 'cleaned_temp.csv')
APY_CSV = os.path.join(DATA_DIR, 'modified_APY.csv')
SHAPEFILE = os.path.join(DATA_DIR, 'gadm36_IND_3.shp')  # optional
PLOTS_DIR = './plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
# 1. Load datasets
# ---------------------------
print('Loading CSVs...')
df_demand = pd.read_csv(DEMAND_CSV)
df_rain = pd.read_csv(RAINFALL_CSV)
df_temp = pd.read_csv(TEMP_CSV)
df_apy = pd.read_csv(APY_CSV)
print('Loaded: ', [len(df) for df in [df_demand, df_rain, df_temp, df_apy]])


Loading CSVs...


FileNotFoundError: [Errno 2] No such file or directory: '/content/modified_agriculture.csv'

In [ ]:
# ---------------------------
# 2. Normalize & Clean
# ---------------------------
for df in [df_demand, df_rain, df_temp, df_apy]:
    df.columns = df.columns.str.strip().str.lower()

# Basic tidy-up
df_rain['district'] = df_rain['district'].astype(str).str.replace('"', '').str.strip().str.lower()
df_rain['month'] = df_rain['month'].astype(str).str.strip().str.upper()

df_temp['month'] = df_temp['month'].astype(str).str.strip().str.upper()

df_demand['district'] = df_demand.get('district', df_demand.get('state')).astype(str).str.strip().str.lower()
df_demand['commodity'] = df_demand['commodity'].astype(str).str.strip().str.lower()
df_demand['state'] = df_demand['state'].astype(str).str.strip().str.lower()

df_apy['district'] = df_apy['district'].astype(str).str.strip().str.lower()
df_apy['crop'] = df_apy['crop'].astype(str).str.strip().str.lower()

# month mappings
month_abbr_to_num = {
    'JAN':1, 'FEB':2, 'MAR':3, 'APR':4, 'MAY':5, 'JUN':6,
    'JUL':7, 'AUG':8, 'SEP':9, 'OCT':10, 'NOV':11, 'DEC':12
}
month_num_to_name = {v:k.title() for k,v in month_abbr_to_num.items()}

# map months
df_rain['month_num'] = df_rain['month'].map(month_abbr_to_num)
df_temp['month_num'] = df_temp['month'].map(month_abbr_to_num)

# If temperature rows have 'avg_temperature' column name variations, standardize
if 'avg_temperature' not in df_temp.columns:
    possible = [c for c in df_temp.columns if 'temp' in c or 'temperature' in c]
    if possible:
        df_temp = df_temp.rename(columns={possible[0]:'avg_temperature'})

# Ensure rainfall column exists
if 'avg_rainfall' not in df_rain.columns and 'rainfall' in df_rain.columns:
    df_rain = df_rain.rename(columns={'rainfall':'avg_rainfall'})


In [ ]:
# ---------------------------
# 3. Prepare weather frame
# ---------------------------
# Average temperature by month (global average)
temp_monthly = df_temp.groupby('month_num')['avg_temperature'].mean().reset_index()

weather = pd.merge(df_rain, temp_monthly, on='month_num', how='left')
# Keep columns: district, month_num, avg_rainfall, avg_temperature
weather = weather.rename(columns={'month_num':'month'})
weather = weather[['district','month','avg_rainfall','avg_temperature']]

In [ ]:
# ---------------------------
# 4. Merge demand with APY and weather (fixed & robust)
# ---------------------------

def fix_duplicate_columns(df):
    """Rename duplicate columns in a DataFrame to make them unique."""
    seen = {}
    new_cols = []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

def ensure_numeric_month(df):
    """Ensure 'month' column exists and is numeric Int64."""
    month_cols = [col for col in df.columns if col.lower().startswith('month')]
    if not month_cols:
        # Try to create from arrival_date if present
        if 'arrival_date' in df.columns:
            df['month'] = pd.to_datetime(df['arrival_date'], errors='coerce').dt.month
    else:
        # Keep only the first month column
        if len(month_cols) > 1:
            for col in month_cols[1:]:
                df = df.drop(columns=[col])
        if month_cols[0] != 'month':
            df = df.rename(columns={month_cols[0]: 'month'})
        # Convert to numeric
        df['month'] = pd.to_numeric(df['month'], errors='coerce').astype('Int64')
    return df

# Step 1: Fix duplicate column names
df_demand = fix_duplicate_columns(df_demand)
df_apy = fix_duplicate_columns(df_apy)
weather = fix_duplicate_columns(weather)

# Step 2: Ensure 'month' column is numeric in demand & weather
df_demand = ensure_numeric_month(df_demand)
weather = ensure_numeric_month(weather)

# Step 3: Merge demand with APY
merged_crop = pd.merge(
    df_demand,
    df_apy,
    left_on=['district', 'commodity'],
    right_on=['district', 'crop'],
    how='left'
)

# Step 4: Merge with weather (avoid duplicate name conflicts)
merged = pd.merge(
    merged_crop,
    weather,
    on=['district', 'month'],
    how='left',
    suffixes=('', '_weather')
)

print('Rows after merge:', len(merged))


In [ ]:
# ---------------------------
# 5. Feature engineering
# ---------------------------

# Drop rows missing key target
merged = merged.dropna(subset=['profit_margin'])

# Season mapping
def month_to_season(m):
    if m in [6, 7, 8, 9]:
        return 'kharif'
    elif m in [10, 11, 12, 1, 2]:
        return 'rabi'
    else:
        return 'zaid'

merged['season'] = merged['month'].apply(
    lambda x: month_to_season(int(x)) if pd.notna(x) else np.nan
)

# Fill missing weather data with month medians
for col in ['avg_rainfall', 'avg_temperature']:
    if col in merged.columns:
        merged[col] = merged.groupby('month')[col].transform(lambda x: x.fillna(x.median()))
        merged[col] = merged[col].fillna(merged[col].median())
    else:
        merged[col] = np.nan

# Base columns (only keep if they exist)
base_cols = ['district', 'month', 'commodity', 'profit_margin', 'avg_rainfall', 'avg_temperature', 'season']
if 'state' in merged.columns:
    base_cols.insert(1, 'state')  # insert state after district

# Build model_df
model_df = merged[base_cols].copy()

# Add APY features if available
for col in ['area', 'yield']:
    model_df[col] = merged[col] if col in merged.columns else np.nan

# Drop rows missing commodity
model_df = model_df.dropna(subset=['commodity'])

# Lowercase & strip strings
for col in ['commodity', 'district', 'season']:
    if col in model_df.columns:
        model_df[col] = model_df[col].astype(str).str.strip().str.lower()

print('Rows prepared for modelling:', len(model_df))
print('Columns in model_df:', model_df.columns.tolist())


In [ ]:
# ---------------------------
# 6. Encoding and train-test
# ---------------------------
le_district = LabelEncoder()
le_crop = LabelEncoder()
le_season = LabelEncoder()

model_df['district_enc'] = le_district.fit_transform(model_df['district'].astype(str))
model_df['crop_enc'] = le_crop.fit_transform(model_df['commodity'].astype(str))
model_df['season_enc'] = le_season.fit_transform(model_df['season'].astype(str))

# numeric fillers
model_df['area'] = model_df['area'].fillna(model_df['area'].median())
model_df['yield'] = model_df['yield'].fillna(model_df['yield'].median())

features = ['district_enc','month','avg_rainfall','avg_temperature','season_enc','crop_enc','area','yield']
X = model_df[features].astype(float)
y = model_df['profit_margin'].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Training set rows:', len(X_train), 'Test set rows:', len(X_test))


In [ ]:
# ---------------------------
# 7. Train RandomForest model (with data cleaning)
# ---------------------------

# Remove rows where target is NaN or infinite
mask = (~y_train.isna()) & np.isfinite(y_train)
X_train_clean = X_train[mask]
y_train_clean = y_train[mask]

model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train_clean, y_train_clean)

# Save model & encoders for later use
joblib.dump(model, 'rf_profit_model.joblib')
joblib.dump(le_district, 'le_district.joblib')
joblib.dump(le_crop, 'le_crop.joblib')
joblib.dump(le_season, 'le_season.joblib')

print('Model and encoders saved.')


In [ ]:
# ---------------------------
# 8. Recommendation function
# ---------------------------

# Helper to get average weather for district/month
weather_lookup = weather.set_index(['district', 'month'])[['avg_rainfall', 'avg_temperature']]

# Nearest-district fallback using GeoPandas if available
india_gdf = None
if GEO_ENABLED and os.path.exists(SHAPEFILE):
    india_gdf = gpd.read_file(SHAPEFILE)
    india_gdf = india_gdf.to_crs(epsg=3857)  # project to meters for distance calc
    india_gdf['NAME_2'] = india_gdf['NAME_2'].astype(str).str.strip().str.lower()

def find_nearest_district(user_district):
    if india_gdf is None:
        return None
    user_district = user_district.strip().lower()
    if user_district not in india_gdf['NAME_2'].values:
        return None
    user_geom = india_gdf.loc[india_gdf['NAME_2'] == user_district, 'geometry'].values[0]
    india_gdf['dist_to_user'] = india_gdf['geometry'].distance(user_geom)
    nearest = india_gdf[india_gdf['NAME_2'] != user_district].sort_values('dist_to_user').iloc[0]
    return nearest['NAME_2']

def recommend_crops_ai(district, month, top_n=5):
    """
    district: district name (string)
    month: integer month (1-12) or month name (e.g., 'July')
    returns: list of tuples (crop, predicted_profit_percentage)
    """
    # Normalize month
    if isinstance(month, str):
        month = month.strip()[:3].upper()
        month = month_abbr_to_num.get(month)
    if pd.isna(month) or month is None:
        raise ValueError('Invalid month')

    district_key = district.strip().lower()

    # Get weather
    try:
        rainfall, temp = weather_lookup.loc[(district_key, int(month))]
    except KeyError:
        # Fallback to nearest district
        nearest = find_nearest_district(district_key)
        if nearest:
            try:
                rainfall, temp = weather_lookup.loc[(nearest, int(month))]
                district_key = nearest
            except KeyError:
                rainfall = merged['avg_rainfall'].median()
                temp = merged['avg_temperature'].median()
        else:
            rainfall = merged['avg_rainfall'].median()
            temp = merged['avg_temperature'].median()

    # Force numeric conversion
    rainfall = pd.to_numeric(rainfall, errors='coerce')
    temp = pd.to_numeric(temp, errors='coerce')

    # Fallback if still NaN
    if pd.isna(rainfall):
        rainfall = merged['avg_rainfall'].median()
    if pd.isna(temp):
        temp = merged['avg_temperature'].median()

    season = month_to_season(int(month))

    # Predict for all crops
    crops = list(le_crop.classes_)
    preds = []
    for crop in crops:
        try:
            d_enc = le_district.transform([district_key])[0]
        except ValueError:
            # fallback: nearest district or first available
            nearest = find_nearest_district(district_key)
            if nearest in le_district.classes_:
                d_enc = le_district.transform([nearest])[0]
            else:
                d_enc = 0

        c_enc = le_crop.transform([crop])[0]
        s_enc = le_season.transform([season])[0] if season in le_season.classes_ else 0

        feat = np.array([[d_enc, int(month), float(rainfall), float(temp),
                          s_enc, c_enc,
                          float(model_df['area'].median()),
                          float(model_df['yield'].median())]])

        pred_value = model.predict(feat)[0]
        preds.append((crop, float(pred_value) * 100))  # keep as percentage

    preds = sorted(preds, key=lambda x: x[1], reverse=True)
    return preds[:top_n]

# ---------------------------
# User Input
# ---------------------------
user_district = input("Enter district name: ").strip()
user_month = input("Enter month (number 1-12 or name like 'July'): ").strip()
user_year = input("Enter year (e.g., 2024): ").strip()

try:
    user_year = int(user_year)
except ValueError:
    from datetime import datetime
    print("Invalid year entered. Defaulting to current year.")
    user_year = datetime.now().year

# ---------------------------
# Make recommendation
# ---------------------------
try:
    recommendations = recommend_crops_ai(user_district, user_month, top_n=5)

    # Determine month number for season calculation
    if user_month.isdigit():
        month_num = int(user_month)
    else:
        month_num = month_abbr_to_num.get(user_month[:3].upper())

    season_for_user = month_to_season(month_num)

    # Create demand priority mapping
    demand_priority = {"High": 3, "Normal": 2, "Low": 1}

    # All crops with demand info
    merged['demand_rank'] = merged['demand'].map(demand_priority).fillna(0)

    # Prepare ranked subcrops from other seasons
    other_season_crops = merged[
        (merged['season'] != season_for_user) &
        (merged['demand_rank'] >= 2)
    ].copy()

    ranked_subcrops = (
        other_season_crops.groupby('crop')['demand_rank']
        .max().sort_values(ascending=False)
    )
    subcrop_list = list(ranked_subcrops.index)  # ordered list

    table_data = []
    for i, (main_crop, profit_pct) in enumerate(recommendations):
        sub_crop = "No alternative found"
        demand_level = "Normal"

        if subcrop_list:
            # round-robin: reuse subcrops if we run out
            candidate = subcrop_list[i % len(subcrop_list)]
            if candidate != main_crop:
                sub_crop = candidate
                demand_level = other_season_crops.loc[
                    other_season_crops['crop'] == sub_crop, 'demand'
                ].iloc[0]

        table_data.append({
            'Main Crop': main_crop,
            'Suggested Sub-Crop': sub_crop,
            'Predicted Profit (%)': round(profit_pct/100),
            'Sub-Crop Demand': demand_level
        })

    print(f"\nTop 5 crop recommendations for {user_district.title()} in {user_month} {user_year}:")
    print(pd.DataFrame(table_data))

except Exception as e:
    print("Error making recommendations:", e)
    import traceback
    traceback.print_exc()


In [ ]:
# ---------------------------
# 9. Visualizations (6 required)
# ---------------------------
sns.set(style='whitegrid')

# 1) Yield vs Year for selected crop (using APY data)


# Load APY dataset
df_apy = pd.read_csv(APY_CSV)

import os
import difflib
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


def plot_yield_vs_year_for_top_crop(preds, model_df, df=df_apy, save_dir=PLOTS_DIR):
    """
    Plot yield vs year for the top recommended crop
    with fuzzy matching if exact crop not found in APY dataset.
    """
    try:
        # --------------------------
        # 1. Safely choose crop_choice
        # --------------------------
        crop_choice = None
        if preds is not None and isinstance(preds, (list, tuple)) and len(preds) > 0: # Added isinstance check
                first = preds[0]
                if isinstance(first, (list, tuple)) and len(first) > 0:
                    crop_choice = first[0]
                elif isinstance(first, str):
                    crop_choice = first
        elif isinstance(preds, str): # Handle case where preds is just a crop string
             crop_choice = preds

        if not crop_choice:
            crop_choice = model_df['commodity'].mode().iloc[0]

        crop_choice = str(crop_choice).strip()
        print(f"🔍 Selected crop: {crop_choice}")

        # --------------------------
        # 2. Try exact match
        # --------------------------
        crop_df = df[df['Crop'].str.lower() == crop_choice.lower()]

        # --------------------------
        # 3. If not found → fuzzy match
        # --------------------------
        if crop_df.empty:
            all_crops = df['Crop'].dropna().unique().tolist()
            close_matches = difflib.get_close_matches(crop_choice, all_crops, n=1, cutoff=0.6)
            if close_matches:
                matched_crop = close_matches[0]
                print(f"⚠️ No exact match. Using closest match: {matched_crop}")
                crop_df = df[df['Crop'].str.lower() == matched_crop.lower()]
                crop_choice = matched_crop
            else:
                print(f"❌ No APY data found for crop: {crop_choice}")
                return

        # --------------------------
        # 4. Plot yield vs year
        # --------------------------
        plt.figure(figsize=(10,6))
        sns.lineplot(data=crop_df, x='Crop_Year', y='Yield', marker="o")
        plt.xlabel('Year')
        plt.ylabel('Yield (tons/hectare)')
        plt.title(f'Yield vs Year for {crop_choice.title()}')
        plt.tight_layout()

        # Save file
        os.makedirs(save_dir, exist_ok=True)
        output_file = os.path.join(save_dir, f'yield_vs_year_{crop_choice.lower().replace(" ","_")}.png')
        plt.savefig(output_file, dpi=300)
        plt.close()

        print(f"✅ Plot saved: {output_file}")

    except Exception as e:
        print(f"❌ Error plotting yield vs year: {e}")

plot_yield_vs_year_for_top_crop(preds, model_df)


# 2) Seasonal Profitability Heatmap (crop x season)
pivot = model_df.pivot_table(index='commodity', columns='season', values='profit_margin', aggfunc='mean').fillna(0)
plt.figure(figsize=(12,10))
sns.heatmap(pivot, cmap='YlGnBu')
plt.title('Average Profit Margin by Crop and Season')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'profit_heatmap_season_crop.png'))
plt.close()

# 3) Rainfall vs Profit Margin (scatter + reg)
plt.figure(figsize=(8,6))
sns.regplot(data=model_df, x='avg_rainfall', y='profit_margin', scatter_kws={'alpha':0.4})
plt.title('Rainfall vs Profit Margin')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'rainfall_vs_profit.png'))
plt.close()

# 4) Temperature vs Profit Margin (scatter + reg)
plt.figure(figsize=(8,6))
sns.regplot(data=model_df, x='avg_temperature', y='profit_margin', scatter_kws={'alpha':0.4})
plt.title('Temperature vs Profit Margin')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'temperature_vs_profit.png'))
plt.close()

# 5) Model Predictions: Top crops for user's selected district+month (bar)
try:
    # Convert month to numeric if it's a string
    if isinstance(user_month, str):
        month_num = month_abbr_to_num.get(user_month.strip()[:3].upper())
        if month_num is None:
            raise ValueError("Invalid month name")
        user_month = month_num

    preds = recommend_crops_ai(user_district, user_month, top_n=8)
    pred_df = pd.DataFrame(preds, columns=['crop','predicted_profit'])

    plt.figure(figsize=(8,6))
    sns.barplot(data=pred_df, x='predicted_profit', y='crop')

    # Format month name for display
    month_name = month_num_to_name.get(int(user_month), str(user_month))

    plt.title(f'Top Predicted Crops for {user_district.title()} in {month_name}')
    plt.xlabel('Predicted Profit Margin (%)')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'user_predicted_top_crops.png'))
    plt.close()
except Exception as e:
    print(f"Error generating prediction plot for user inputs: {e}")
    # Fallback to representative district/month
    rep_district = model_df['district'].mode().iloc[0]
    rep_month = int(model_df['month'].mode().iloc[0])
    preds = recommend_crops_ai(rep_district, rep_month, top_n=8)
    pred_df = pd.DataFrame(preds, columns=['crop','predicted_profit'])

    plt.figure(figsize=(8,6))
    sns.barplot(data=pred_df, x='predicted_profit', y='crop')
    plt.title(f'Top Predicted Crops for {rep_district.title()} in {month_num_to_name[rep_month]}')
    plt.xlabel('Predicted Profit Margin (%)')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'predicted_top_crops.png'))
    plt.close()

# 6) Geographical Choropleth for a chosen crop (if geopandas available)
# Use the top recommended crop from user's prediction if available
try:
    if len(preds) > 0:
        crop_choice = preds[0][0]  # Use top recommended crop
    else:
        crop_choice = model_df['commodity'].mode().iloc[0]
except:
    crop_choice = model_df['commodity'].mode().iloc[0]

if GEO_ENABLED and os.path.exists(SHAPEFILE):
    gdf = gpd.read_file(SHAPEFILE)
    gdf['NAME_2'] = gdf['NAME_2'].astype(str).str.strip().str.lower()
    crop_profit = model_df[model_df['commodity']==crop_choice].groupby('district')['profit_margin'].mean().reset_index()
    gdf = gdf.merge(crop_profit, left_on='NAME_2', right_on='district', how='left')
    fig, ax = plt.subplots(1,1, figsize=(12,10))
    gdf.plot(column='profit_margin', cmap='YlOrBr', legend=True, ax=ax, missing_kwds={"color": "lightgrey"})
    ax.set_title(f'Average Profit Margin for {crop_choice.title()} by District')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'choropleth_crop_profit.png'))
    plt.close()
else:
    # If geopandas not available, create a district-level horizontal bar chart instead
    crop_profit = model_df[model_df['commodity']==crop_choice].groupby('district')['profit_margin'].mean().sort_values(ascending=False).head(20).reset_index()
    plt.figure(figsize=(10,8))
    sns.barplot(data=crop_profit, x='profit_margin', y='district')
    plt.title(f'Top 20 Districts by Average Profit for {crop_choice.title()}')
    plt.xlabel('Average Profit Margin (%)')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'top_districts_crop_profit.png'))
    plt.close()

print('All plots saved to', PLOTS_DIR)

In [ ]:
# ---------------------------
# 10. Save artifacts (sample data & user predictions)
# ---------------------------

# Define output directory if not already defined
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = './outputs'
    os.makedirs(OUTPUT_DIR, exist_ok=True) # Ensure directory exists

# Load forecast data
try:
    df_rain = pd.read_csv(RAINFALL_CSV)
    df_temp = pd.read_csv(TEMP_CSV)
    print(f"Weather forecast data loaded from {RAINFALL_CSV} and {TEMP_CSV}")

    # Clean columns for consistency
    df_rain['district'] = df_rain['district'].str.replace('"', '').str.strip()
    df_temp['district'] = df_temp['location'].str.strip()
    df_temp['month_num'] = pd.to_datetime(df_temp['Month'], format='%b').dt.month

    # Tamil Nadu state-level average temperatures for fallback
    tn_temp_avg = df_temp.groupby('month_num')['Avg_Temperature'].mean().to_dict()

    # Create combined district list from both dataframes
    rain_districts = set(df_rain['district'].str.lower().unique())
    temp_districts = set(df_temp['district'].str.lower().unique())
    all_forecast_districts = list(rain_districts.union(temp_districts))
except Exception as e:
    print(f"Error loading forecast data: {e}")
    df_rain = pd.DataFrame()
    df_temp = pd.DataFrame()
    all_forecast_districts = []
    tn_temp_avg = {}

def find_nearest_district(target_district):
    """Find the nearest district with available forecast data"""
    if not all_forecast_districts:
        return None

    target_district = target_district.strip().lower()

    # If exact match exists
    if target_district in all_forecast_districts:
        return target_district

    # Find closest match by string similarity
    try:
        from difflib import get_close_matches
        matches = get_close_matches(target_district, all_forecast_districts, n=1, cutoff=0.6)
        if matches:
            return matches[0]
    except:
        pass

    # If no close match, return first available district
    return all_forecast_districts[0] if all_forecast_districts else None

def get_weather_forecast(district, month):
    """Get rainfall and temperature forecast for district and month with fallback"""
    try:
        district_lower = district.strip().lower()
        month_num = int(month) if isinstance(month, (int, float)) else month_abbr_to_num.get(month.strip()[:3].upper())

        # Try rainfall from district
        rain = None
        try:
            rain = df_rain[(df_rain['district'].str.lower() == district_lower) &
                           (df_rain['month'] == month_num)]['avg_rainfall'].mean()
        except:
            pass

        # Try temperature from district
        temp = None
        temp_source = None
        try:
            temp_val = df_temp[(df_temp['district'].str.lower() == district_lower) &
                               (df_temp['month_num'] == month_num)]['Avg_Temperature']
            if not temp_val.empty:
                temp = temp_val.mean()
                temp_source = district.title()
        except:
            pass

        # Fallback: Tamil Nadu avg temperature if district temp not available
        if temp is None and month_num in tn_temp_avg:
            temp = tn_temp_avg[month_num]
            temp_source = "Celsius"

        return rain, temp, temp_source
    except Exception:
        return None, None, None

try:
    # Save sample data
    model_df.head(200).to_csv(os.path.join(OUTPUT_DIR, 'model_data_sample.csv'), index=False)

    # Save user predictions
    user_preds = recommend_crops_ai(user_district, user_month, top_n=50)
    pd.DataFrame(user_preds, columns=['crop','predicted_profit']).to_csv(
        os.path.join(OUTPUT_DIR, 'user_predictions.csv'), index=False)

    print(f'Artifacts written to {OUTPUT_DIR}: model_data_sample.csv, user_predictions.csv')
except Exception as e:
    print(f"Error saving artifacts: {e}")

# ---------------------------
# 11. Enhanced summary function with weather forecast
# ---------------------------

def summary_for_district(district, month):
    """
    Return analysis summary for a district + month with weather forecast

    Args:
        district: District name (str)
        month: Month (int or str)

    Returns:
        Formatted string summary
    """
    try:
        # Convert month if needed
        month_num = month
        if isinstance(month, str):
            month_num = month_abbr_to_num.get(month.strip()[:3].upper())
            if month_num is None:
                raise ValueError("Invalid month format")

        # Get weather data
        weather_info = "\nWeather Data:"
        forecast_rain, forecast_temp, temp_source = get_weather_forecast(district, month_num)

        if forecast_rain is not None:
            weather_info += f"\n- Rainfall: {forecast_rain:.1f} mm"
        else:
            weather_info += "\n- Rainfall: Not available"

        if forecast_temp is not None:
            source_note = f" ({temp_source})" if temp_source else ""
            weather_info += f"\n- Temperature: {forecast_temp:.1f}°C{source_note}"
        else:
            weather_info += "\n- Temperature: Not available"

        # Get predictions
        preds = recommend_crops_ai(district, month_num, top_n=10)
        top_preds = '\n'.join([f"{i+1}. {c.title():<15} {p//100}%" for i,(c,p) in enumerate(preds)])

        return (f"Agricultural Summary\n{'='*30}\n"
                f"District: {district.title()}\n"
                f"Month: {month_num_to_name.get(month_num, month_num)}"
                f"{weather_info}\n\n"
                f"Top Recommended Crops for {district.title()}:\n{top_preds}")

    except Exception as e:
        return f"Error generating summary: {str(e)}"

# Generate and print user summary
user_summary = summary_for_district(user_district, user_month)
print('\n=== Agricultural Recommendation ===')
print(user_summary)

# Save the summary to a text file
try:
    with open(os.path.join(OUTPUT_DIR, 'agricultural_recommendation.txt'), 'w') as f:
        f.write(user_summary)
    print(f'\nRecommendation saved to {OUTPUT_DIR}/agricultural_recommendation.txt')
except Exception as e:
    print(f"\nCould not save recommendation file: {e}")

print('\nAnalysis complete. Check outputs in:')
print(f"- Plots: {PLOTS_DIR}")
print(f"- Data: {OUTPUT_DIR}")

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

def plot_district_weather(district, month, shapefile=SHAPEFILE):
    """
    Plot the Indian map highlighting the given district,
    annotate with weather forecast + add summary block on left side
    """
    try:
        # Load India shapefile
        india = gpd.read_file(shapefile)
        india['district'] = india['NAME_3'].str.strip().str.lower()

        # Find the matching district
        target_district = find_nearest_district(district)
        if not target_district:
            print("District not found in shapefile or forecast data")
            return

        # Filter district geometry
        district_geom = india[india['district'] == target_district]
        if district_geom.empty:
            print(f"District {district} not found in shapefile")
            return

        # Get weather forecast
        forecast_rain, forecast_temp, temp_source = get_weather_forecast(district, month)

        # Get crop recommendations
        preds = recommend_crops_ai(district, month, top_n=5)
        crop_text = "\n".join([f"{i+1}. {c.title()} ({p//100}%)" for i,(c,p) in enumerate(preds)])

        # Create base map
        fig, ax = plt.subplots(figsize=(14, 10))
        india.boundary.plot(ax=ax, linewidth=0.5, color="gray")

        # Highlight district
        district_geom.boundary.plot(ax=ax, color="red", linewidth=2)
        district_geom.plot(ax=ax, color="lightblue", alpha=0.5)

        # District centroid (for arrow pointing)
        centroid = district_geom.geometry.centroid.iloc[0]

        # Left-side summary text
        summary_text = (
            f"Agricultural Summary\n{'='*25}\n"
            f"District: {district.title()}\n"
            f"Month: {month_num_to_name.get(month, month)}\n\n"
            f"Weather Data:\n"
            f"- Rainfall: {forecast_rain:.1f} mm\n"
            f"- Temperature: {forecast_temp:.1f}°C\n\n"
            f"Top Crops:\n{crop_text}"
        )

        # Add text block on LEFT side
        ax.text(
            x=ax.get_xlim()[0] - 5,   # push to left
            y=ax.get_ylim()[1],       # top of map
            s=summary_text,
            fontsize=10,
            ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.8)
        )

        # Arrow annotation to district
        ax.annotate(
            district.title(),
            xy=(centroid.x, centroid.y),
            xytext=(centroid.x+2, centroid.y+2),
            fontsize=11,
            color="black",
            arrowprops=dict(facecolor='black', arrowstyle="->")
        )

        plt.title(f"Forecast & Crop Recommendations for {district.title()} - {month_num_to_name.get(month, month)}", fontsize=14)
        plt.axis("off")
        plt.tight_layout()

        plt.show()

    except Exception as e:
        print(f"Error plotting district weather: {e}")

# Example usage
plot_district_weather(user_district, user_month)


In [ ]:
# ---------------------------
# 4. Save and display HTML
# ---------------------------

output_html_file = os.path.join(OUTPUT_DIR, 'agricultural_dashboard.html')

with open(output_html_file, 'w') as f:
    f.write(full_html)

print(f"Interactive dashboard saved to: {output_html_file}")
print("\nTo view the dashboard:")
print(f"1. Download the file '{output_html_file}' from the file browser (left panel).")
print("2. Open the downloaded HTML file in your web browser.")

# ---------------------------
# 5. Finish task
# ---------------------------

print("\n=== Task Complete ===")
print("An interactive agricultural insights dashboard has been generated.")
print(f"It includes visualizations for yield vs year, seasonal profitability, rainfall vs profit, temperature vs profit, and top predicted crops.")
if GEO_ENABLED and os.path.exists(SHAPEFILE):
    print("A choropleth map showing average profit margin by district is also included.")
else:
    print("A bar chart showing top districts by average profit is included instead of a choropleth map.")
print(f"The dashboard HTML file is saved at {output_html_file}.")

print("\nNext Steps:")
print("- Explore the interactive plots in the HTML dashboard.")
print("- Share the HTML file with others.")
print("- Consider deploying the dashboard on a web server for broader access.")
print("- If you have more data, we can update the dashboard with the new information.")
print("- We could also explore other visualization types or add more features to the dashboard.")

In [ ]:
# ---------------------------
# 4. Save and display HTML
# ---------------------------

output_html_file = os.path.join(OUTPUT_DIR, 'agricultural_dashboard.html')

with open(output_html_file, 'w') as f:
    f.write(full_html)

print(f"Interactive dashboard saved to: {output_html_file}")
print("\nTo view the dashboard:")
print(f"1. Download the file '{output_html_file}' from the file browser (left panel).")
print("2. Open the downloaded HTML file in your web browser.")

# ---------------------------
# 5. Finish task
# ---------------------------

print("\n=== Task Complete ===")
print("An interactive agricultural insights dashboard has been generated.")
print(f"It includes visualizations for yield vs year, seasonal profitability, rainfall vs profit, temperature vs profit, and top predicted crops.")
if GEO_ENABLED and os.path.exists(SHAPEFILE):
    print("A choropleth map showing average profit margin by district is also included.")
else:
    print("A bar chart showing top districts by average profit is included instead of a choropleth map.")
print(f"The dashboard HTML file is saved at {output_html_file}.")

print("\nNext Steps:")
print("- Explore the interactive plots in the HTML dashboard.")
print("- Share the HTML file with others.")
print("- Consider deploying the dashboard on a web server for broader access.")
print("- If you have more data, we can update the dashboard with the new information.")
print("- We could also explore other visualization types or add more features to the dashboard.")

In [ ]:
# ---------------------------
# 3. Embed plots in HTML
# ---------------------------

# Convert Plotly figures to HTML div strings
# Use config={'displayModeBar': False} to reduce clutter
yield_html = yield_fig.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False}) if yield_fig else ""
heatmap_html = heatmap_fig.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False}) if heatmap_fig else ""
rainfall_html = rainfall_fig.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False}) if rainfall_fig else ""
temperature_html = temperature_fig.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False}) if temperature_fig else ""
predictions_html = predictions_fig.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False}) if predictions_fig else ""
choropleth_html = choropleth_fig.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False}) if choropleth_fig else ""

# Insert plot HTML into the template
# Use format to insert user inputs and plot divs
full_html = html_template.format(
    district=user_district.title(),
    month=month_num_to_name.get(int(user_month), str(user_month)),
    year=user_year,
    yield_plot=yield_html,
    heatmap_plot=heatmap_html,
    rainfall_plot=rainfall_html,
    temperature_plot=temperature_html,
    predictions_plot=predictions_html,
    choropleth_plot=choropleth_html
)

# Need to inject the plot divs into the correct places within the script tag
# A simple way is to replace the placeholder divs directly
full_html = full_html.replace('<div class="plot" id="yield_vs_year"></div>', yield_html)
full_html = full_html.replace('<div class="plot" id="profit_heatmap"></div>', heatmap_html)
full_html = full_html.replace('<div class="plot" id="rainfall_vs_profit"></div>', rainfall_html)
full_html = full_html.replace('<div class="plot" id="temperature_vs_profit"></div>', temperature_html)
full_html = full_html.replace('<div class="plot" id="predictions"></div>', predictions_html)
full_html = full_html.replace('<div class="plot" id="choropleth"></div>', choropleth_html)


print("Plots embedded into HTML template.")

In [ ]:
# ---------------------------
# 2. Create HTML structure
# ---------------------------
html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>Agricultural Dashboard</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{ font-family: sans-serif; }}
        .container {{ display: flex; flex-wrap: wrap; justify-content: space-around; }}
        .plot {{ width: 48%; margin-bottom: 20px; }}
        h1, h2 {{ text-align: center; }}
    </style>
</head>
<body>
    <h1>Agricultural Insights Dashboard</h1>
    <h2>Recommendations for {district} in {month} {year}</h2>

    <div class="container">
        <div class="plot" id="yield_vs_year"></div>
        <div class="plot" id="profit_heatmap"></div>
        <div class="plot" id="rainfall_vs_profit"></div>
        <div class="plot" id="temperature_vs_profit"></div>
        <div class="plot" id="predictions"></div>
        <div class="plot" id="choropleth"></div>
    </div>

    <script>
        // Plotly plots will be embedded here by the Python code
    </script>
</body>
</html>
"""

print("HTML template created.")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# 1) Yield vs Year for selected crop (interactive)
def plot_yield_vs_year_interactive(preds, model_df, df=df_apy):
    try:
        crop_choice = None
        if preds is not None and isinstance(preds, (list, tuple)) and len(preds) > 0:
            first = preds[0]
            if isinstance(first, (list, tuple)) and len(first) > 0:
                crop_choice = first[0]
            elif isinstance(first, str):
                crop_choice = first
        elif isinstance(preds, str):
             crop_choice = preds

        if not crop_choice:
            crop_choice = model_df['commodity'].mode().iloc[0]

        crop_choice = str(crop_choice).strip()

        crop_df = df[df['Crop'].astype(str).str.lower() == crop_choice.lower()]

        if crop_df.empty:
            all_crops = df['Crop'].dropna().unique().tolist()
            close_matches = difflib.get_close_matches(crop_choice, all_crops, n=1, cutoff=0.6)
            if close_matches:
                matched_crop = close_matches[0]
                crop_df = df[df['Crop'].astype(str).str.lower() == matched_crop.lower()]
                crop_choice = matched_crop
            else:
                print(f"❌ No APY data found for crop: {crop_choice}")
                return None # Return None if no data found

        fig = px.line(crop_df, x='Crop_Year', y='Yield', title=f'Yield vs Year for {crop_choice.title()}')
        return fig # Return the plotly figure

    except Exception as e:
        print(f"❌ Error plotting interactive yield vs year: {e}")
        return None

# 2) Seasonal Profitability Heatmap (interactive)
def plot_profit_heatmap_interactive(model_df):
    try:
        pivot = model_df.pivot_table(index='commodity', columns='season', values='profit_margin', aggfunc='mean').fillna(0)
        # Corrected heatmap function call
        fig = go.Figure(data=go.Heatmap(
                   z=pivot.values,
                   x=pivot.columns,
                   y=pivot.index,
                   colorscale='YlGnBu'))
        fig.update_layout(title='Average Profit Margin by Crop and Season')
        return fig
    except Exception as e:
        print(f"❌ Error plotting interactive heatmap: {e}")
        return None

# 3) Rainfall vs Profit Margin (interactive)
def plot_rainfall_vs_profit_interactive(model_df):
    try:
        fig = px.scatter(model_df, x='avg_rainfall', y='profit_margin', title='Rainfall vs Profit Margin', opacity=0.4)
        return fig
    except Exception as e:
        print(f"❌ Error plotting interactive rainfall vs profit: {e}")
        return None

# 4) Temperature vs Profit Margin (interactive)
def plot_temperature_vs_profit_interactive(model_df):
    try:
        fig = px.scatter(model_df, x='avg_temperature', y='profit_margin', title='Temperature vs Profit Margin', opacity=0.4)
        return fig
    except Exception as e:
        print(f"❌ Error plotting interactive temperature vs profit: {e}")
        return None

# 5) Model Predictions: Top crops for user's selected district+month (interactive bar)
def plot_user_predictions_interactive(user_district, user_month, model_df):
    try:
        if isinstance(user_month, str):
            month_num = month_abbr_to_num.get(user_month.strip()[:3].upper())
            if month_num is None:
                raise ValueError("Invalid month name")
            user_month_num = month_num
        else:
            user_month_num = user_month

        preds = recommend_crops_ai(user_district, user_month_num, top_n=8)
        pred_df = pd.DataFrame(preds, columns=['crop','predicted_profit'])

        month_name = month_num_to_name.get(int(user_month_num), str(user_month_num))

        fig = px.bar(pred_df, x='predicted_profit', y='crop', orientation='h',
                     title=f'Top Predicted Crops for {user_district.title()} in {month_name}')
        return fig
    except Exception as e:
        print(f"Error generating interactive prediction plot for user inputs: {e}")
        return None


# 6) Geographical Choropleth for a chosen crop (interactive)
def plot_choropleth_interactive(preds, model_df, shapefile=SHAPEFILE):
    try:
        if GEO_ENABLED and os.path.exists(shapefile):
            if len(preds) > 0:
                crop_choice = preds[0][0]
            else:
                crop_choice = model_df['commodity'].mode().iloc[0]

            gdf = gpd.read_file(shapefile)
            gdf['NAME_2'] = gdf['NAME_2'].astype(str).str.strip().str.lower()
            crop_profit = model_df[model_df['commodity']==crop_choice].groupby('district')['profit_margin'].mean().reset_index()
            gdf = gdf.merge(crop_profit, left_on='NAME_2', right_on='district', how='left')

            fig = px.choropleth(
                gdf,
                geojson=gdf.geometry.__geo_interface__,
                locations=gdf.index,
                color='profit_margin',
                hover_name='NAME_2',
                title=f'Average Profit Margin for {crop_choice.title()} by District',
                color_continuous_scale="YlOrBr",
                projection="mercator"
            )
            fig.update_geos(fitbounds="locations", visible=False)
            return fig
        else:
            print("Geopandas not enabled or shapefile not found. Skipping choropleth.")
            return None
    except Exception as e:
        print(f"Error plotting interactive choropleth: {e}")
        return None

# Generate interactive plots
yield_fig = plot_yield_vs_year_interactive(preds, model_df)
heatmap_fig = plot_profit_heatmap_interactive(model_df)
rainfall_fig = plot_rainfall_vs_profit_interactive(model_df)
temperature_fig = plot_temperature_vs_profit_interactive(model_df)
predictions_fig = plot_user_predictions_interactive(user_district, user_month, model_df)
choropleth_fig = plot_choropleth_interactive(preds, model_df)

print("Interactive plots generated.")